# 01 - Setup e Ingestão
Projeto: Voice of Customer — Segmentação de avaliações por assunto e sentimento com LLM.

Objetivos deste notebook:
1. Criar catálogo, schemas e Volume
2. Baixar o dataset (Women's E-Commerce Clothing Reviews) via Kaggle API
3. **Validar que a Foundation Model API funciona** (embeddings + LLM) antes de seguir

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS voc_project;

CREATE SCHEMA IF NOT EXISTS voc_project.bronze;
CREATE SCHEMA IF NOT EXISTS voc_project.silver;
CREATE SCHEMA IF NOT EXISTS voc_project.gold;

CREATE VOLUME IF NOT EXISTS voc_project.bronze.raw_files;

## Ingestão do dataset via Kaggle API
Suba o arquivo de token (kaggle_token.txt ou kaggle.json) para o Volume antes de rodar.

In [0]:
#%restart_python

In [0]:
%pip install kaggle --quiet
%restart_python

In [0]:
import os

with open("/Volumes/workspace/default/tokens/kaggle_token.txt") as f:
    token = f.read().strip()

os.environ["KAGGLE_API_TOKEN"] = token

In [0]:
# import os
# os.environ["KAGGLE_CONFIG_DIR"] = "/Volumes/voc_project/bronze/raw_files"

import kaggle
kaggle.api.authenticate()
print("✅ Autenticado com sucesso")

In [0]:
import shutil

kaggle.api.dataset_download_files(
    "nicapotato/womens-ecommerce-clothing-reviews",
    path="/tmp/reviews",
    unzip=True
)

for f in os.listdir("/tmp/reviews"):
    shutil.copy(f"/tmp/reviews/{f}", f"/Volumes/voc_project/bronze/raw_files/{f}")

print(os.listdir("/Volumes/voc_project/bronze/raw_files"))

In [0]:
df = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv("/Volumes/voc_project/bronze/raw_files/Womens Clothing E-Commerce Reviews.csv"))

print("Linhas:", df.count())
print("Colunas:", df.columns)
display(df.limit(5))

## ⚠️ Teste crítico: Foundation Model API
Antes de seguir, precisamos confirmar que conseguimos:
1. Gerar **embeddings** (para clusterização)
2. Chamar um **LLM generativo** (para sentimento e rótulos)

Se algo falhar aqui, ajustamos o plano antes de investir nas próximas camadas.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

resposta = w.serving_endpoints.query(
    name="databricks-gte-large-en",
    input="This dress fits perfectly and the fabric is amazing."
)

vetor = resposta.data[0].embedding
print("✅ Embedding gerado com sucesso")
print("Dimensões do vetor:", len(vetor))
print("Primeiros 5 valores:", vetor[:5])

In [0]:
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

resposta_llm = w.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        ChatMessage(
            role=ChatMessageRole.USER,
            content="Classify the sentiment of this review as positive, negative, or neutral. "
                    "Answer with only one word.\n\nReview: 'The fabric feels cheap and it fell apart after one wash.'"
        )
    ],
    max_tokens=10
)

print("✅ LLM respondeu com sucesso")
print("Resposta:", resposta_llm.choices[0].message.content)

## Alternativa: testar via SQL (ai_query)
O Databricks também permite chamar modelos direto no SQL. Útil para processar em lote depois.

In [0]:
%sql
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Classify the sentiment as positive, negative, or neutral. Answer with one word only. Review: The dress is beautiful and arrived on time.'
) AS teste_sentimento;

In [0]:
print("=" * 50)
print("SETUP CONCLUÍDO")
print("=" * 50)
print("✅ Catálogo e schemas criados")
print("✅ Dataset baixado:", df.count(), "reviews")
print("✅ Embeddings funcionando (1024 dimensões)")
print("✅ LLM funcionando")
print("\nPronto para a camada Bronze.")